# Module 2: RAG, Audio & Evaluation

**Goal:** Build a multi-modal RAG pipeline. Transcribe audio, chunk text, index in Vector DB, and evaluate retrieval/answer quality.

## 1. Setup
Load keys and initialize clients.

In [1]:
import os
import math
import glob
from typing import List, Dict, Tuple
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

load_dotenv(override=True)
client = OpenAI()

# Constants
EMBEDDING_MODEL = "text-embedding-3-small"
GPT_MODEL = "gpt-4o-mini"
COLLECTION_NAME = "rag_docs"
CHROMA_PATH = "./chroma_db"

print("Clients initialized.")

Clients initialized.


## 2. Audio Transcription (Whisper)
We use OpenAI's Whisper model to transcribe audio files into text for our RAG system.

In [2]:
def transcribe_audio(file_path: str) -> str:
    """Transcribe audio file using OpenAI Whisper."""
    if not os.path.exists(file_path):
        return f"Error: File {file_path} not found. (Skipping transcription)"
    
    print(f"Transcribing {file_path}...")
    with open(file_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", 
            file=audio_file
        )
    return transcript.text

# Example usage (commented out unless you have a file)
# text = transcribe_audio("meeting_recording.mp3")
# print(text[:100])...

## 3. Storage & Indexing (ChromaDB + Chunking)
We need to split text into manageable chunks and store embeddings.

In [3]:
# Initialize Chroma
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name=EMBEDDING_MODEL
)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=openai_ef
)

def naive_chunker(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """Simple overlapping chunker."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def ingest_text(text: str, source_id: str):
    """Chunk and ingest text into Chroma."""
    chunks = naive_chunker(text)
    ids = [f"{source_id}_{i}" for i in range(len(chunks))]
    metadatas = [{"source": source_id} for _ in chunks]
    
    if chunks:
        collection.add(
            documents=chunks,
            ids=ids,
            metadatas=metadatas
        )
        print(f"Ingested {len(chunks)} chunks from {source_id}.")

# Ingest some sample data
sample_text = """
The trajectory of Artificial Intelligence has been marked by distinct eras. 
Early AI (1950s-1980s) focused on symbolic logic and expert systems. 
The Machine Learning era (1990s-2010s) shifted to statistical patterns. 
Now, the Generative AI era (2020s-Present) is driven by Foundation Models and Large Language Models.
RAG (Retrieval Augmented Generation) allow LLMs to access external data.
"""
ingest_text(sample_text, "ai_history_sample")

Ingested 1 chunks from ai_history_sample.


## 4. Retrieval & Generation (The RAG Loop)

In [4]:
def query_rag(question: str, n_results: int = 2) -> Tuple[str, List[str]]:
    """Query the RAG system."""
    # 1. Retrieve
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )
    
    retrieved_docs = results['documents'][0] if results['documents'] else []
    context_str = "\n\n".join(retrieved_docs)
    
    # 2. Augment Prompt
    system_prompt = "You are a helpful assistant. Use the provided context to answer the user's question."
    user_prompt = f"Context:\n{context_str}\n\nQuestion: {question}"
    
    # 3. Generate
    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content, retrieved_docs

# Test
ans, docs = query_rag("What are the eras of AI history?")
print(f"Answer: {ans}\n")
print(f"Retrieved {len(docs)} docs.")

Answer: The history of AI can be divided into three distinct eras:

1. **Early AI (1950s-1980s)**: This era focused on symbolic logic and expert systems.
2. **Machine Learning Era (1990s-2010s)**: During this period, the focus shifted to statistical patterns and machine learning techniques.
3. **Generative AI Era (2020s-Present)**: This current era is characterized by the use of Foundation Models and Large Language Models, with advancements like Retrieval Augmented Generation (RAG) allowing these models to access external data.

Retrieved 1 docs.


## 5. Evaluation (Metrics)
We calculate MRR (Mean Reciprocal Rank) and NDCG to measure retrieval quality.

In [5]:
def calculate_mrr(keyword: str, retrieved_docs: List[str]) -> float:
    """Calculate Reciprocal Rank for a keyword."""
    keyword = keyword.lower()
    for i, doc in enumerate(retrieved_docs, start=1):
        if keyword in doc.lower():
            return 1.0 / i
    return 0.0

def calculate_ndcg(keyword: str, retrieved_docs: List[str], k: int = 10) -> float:
    """Calculate nDCG (Simplified Binary Relevance)."""
    keyword = keyword.lower()
    relevances = [1 if keyword in doc.lower() else 0 for doc in retrieved_docs[:k]]
    
    # DCG
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances))
    
    # IDCG (Ideal: relevances sorted desc)
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevances))
    
    return dcg / idcg if idcg > 0 else 0.0

# Eval Case
test_q = "What relies on statistical patterns?"
test_keyword = "Machine Learning"

_, r_docs = query_rag(test_q)
mrr = calculate_mrr(test_keyword, r_docs)
ndcg = calculate_ndcg(test_keyword, r_docs)

print(f"For keyword '{test_keyword}':")
print(f"MRR: {mrr:.4f}")
print(f"NDCG: {ndcg:.4f}")

For keyword 'Machine Learning':
MRR: 1.0000
NDCG: 1.0000
